# Generation Capacities and Load Settings

In this notebook, we demonstrate how to generate capacity values for generator buses and assign load values for load buses in a synthetic power grid.

In [ ]:
# Install powergrid_synth (--no-deps avoids upgrading Colab's pre-installed
# numpy/scipy, so no runtime restart is needed).
!pip install -q --no-deps git+https://github.com/cookbook-ms/Chung_Lu_Chain-synthesizer.git
# Pin numpy & scipy to the versions already loaded in this runtime, then
# install the remaining dependencies normally.
!pip install -q \
    "numpy==$(python3 -c 'import numpy;print(numpy.__version__)')" \
    "scipy==$(python3 -c 'import scipy;print(scipy.__version__)')" 

## Basics

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

from powergrid_synth import (
    PowerGridGenerator,
    InputConfigurator,
    BusTypeAllocator,
    CapacityAllocator,
    LoadAllocator,
    GenerationDispatcher,
    TransmissionLineAllocator,
    GridVisualizer,
)

## Synthetic raw topology generation & Bus Type Assignment

First, we generate the topology and assign bus types (see previous notebooks for details).

In [ ]:
# Generate topology
level_specs = [
    {'n': 20, 'avg_k': 4.0, 'diam': 6, 'dist_type': 'dgln'},
    {'n': 20, 'avg_k': 3.0, 'diam': 10, 'dist_type': 'dpl'},
    {'n': 10, 'avg_k': 2.0, 'diam': 10, 'dist_type': 'dgln'}
]
connection_specs = {
    (0, 1): {'type': 'k-stars', 'c': 0.174, 'gamma': 4.15},
    (1, 2): {'type': 'k-stars', 'c': 0.15, 'gamma': 4.15}
}

configurator = InputConfigurator(seed=100)
params = configurator.create_params(level_specs, connection_specs)

gen = PowerGridGenerator(seed=100)
grid_graph = gen.generate_grid(
    degrees_by_level=params['degrees_by_level'],
    diameters_by_level=params['diameters_by_level'],
    transformer_degrees=params['transformer_degrees'],
    keep_lcc=True,
)
print(f"Grid Generated: {grid_graph.number_of_nodes()} nodes, {grid_graph.number_of_edges()} edges")

# Assign bus types
allocator = BusTypeAllocator(grid_graph, entropy_model=1)
bus_types = allocator.allocate(max_iter=100, population_size=100)
nx.set_node_attributes(grid_graph, bus_types, name="bus_type")

counts = Counter(bus_types.values())
total = sum(counts.values())
print(f"Generators: {counts['Gen']} ({counts['Gen']/total:.1%})")
print(f"Loads:      {counts['Load']} ({counts['Load']/total:.1%})")
print(f"Connectors: {counts['Conn']} ({counts['Conn']/total:.1%})")

## Generation Capacities

In [ ]:
print("\n[6] Allocating Capacity...")
cap_allocator = CapacityAllocator(grid_graph)
capacities = cap_allocator.allocate()
total_gen = sum(capacities.values())
print(f"Total Generation: {total_gen:.2f} MW")
# Attach to graph
nx.set_node_attributes(grid_graph, capacities, name="pg_max")

In [ ]:
# Check top 10 generators
sorted_gens = sorted(capacities.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Generators by Capacity:")
for node, cap in sorted_gens[:10]:
    print(f"  Node {node}: {cap:.2f} MW (Degree: {grid_graph.degree(node)})")

## Load Settings

In [ ]:
print("\n[7] Allocating Loads ...")
load_allocator = LoadAllocator(grid_graph, ref_sys_id=1)
loads = load_allocator.allocate(loading_level='H')

# Attach to graph (attribute 'pl' for active power load)
nx.set_node_attributes(grid_graph, loads, name="pl")

total_load = sum(loads.values())
print(f"Total Load: {total_load:.2f} MW")

print(f"System Loading: {total_load/total_gen:.1%}")

In [ ]:
# Plot Distribution
load_vals = list(loads.values())

if load_vals:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 2)
    plt.hist(load_vals, bins=30, color='orange', edgecolor='black')
    plt.title("Load Size Distribution")
    plt.xlabel("Load (MW)")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    
# Plot Distribution
caps = list(capacities.values())
if caps:
    plt.subplot(1, 2, 1)
    plt.hist(caps, bins=30, color='skyblue', edgecolor='black')
    plt.title("Generator Capacity Distribution")
    plt.xlabel("Capacity (MW)")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.show()